# DPO 精简版:损失函数 + 数据格式 + 训练配置

> 本 notebook 是第 12 章的速查卡。完整推导见 [`ch12.ipynb`](./ch12.ipynb)。

## DPO 损失函数

$$\mathcal{L}_{\text{DPO}} = -\log \sigma\!\left(\beta \left[\big(\log \pi(y_w) - \log \pi(y_l)\big) - \big(\log \pi_{\text{ref}}(y_w) - \log \pi_{\text{ref}}(y_l)\big)\right]\right)$$

| 符号 | 含义 |
|---|---|
| $y_w$ / $y_l$ | chosen(win)/ rejected(loss)回答 |
| $\pi$ | 当前策略模型(可训练) |
| $\pi_{\text{ref}}$ | 参考模型(冻结的 SFT 副本) |
| $\beta$ | 约束温度(minimind 默认 0.15) |

**直觉**:鼓励策略比参考模型「更偏 chosen」。logits > 0 → loss 小;logits < 0 → loss 大。

## 数据格式

```json
{
  "chosen":   [{"role":"user","content":"..."}, {"role":"assistant","content":"好回答"}],
  "rejected": [{"role":"user","content":"..."}, {"role":"assistant","content":"差回答"}]
}
```

- chosen 和 rejected 共享**相同的 prompt**,只有 assistant 回答不同
- `generate_loss_mask` 只标记 assistant span(mask=1),prompt 和 padding 不计入 loss

In [ ]:
# DPO loss 的 5 行核心实现(train_dpo.py:~34-50 (@67f114a))
import torch
import torch.nn.functional as F

def dpo_loss(ref_lp, policy_lp, mask, beta=0.15):
    ref_lp = (ref_lp * mask).sum(dim=1)
    policy_lp = (policy_lp * mask).sum(dim=1)
    half = ref_lp.shape[0] // 2
    logits = (policy_lp[:half] - policy_lp[half:]) - (ref_lp[:half] - ref_lp[half:])
    return -F.logsigmoid(beta * logits).mean()

## 训练配置对比

| 配置 | SFT | DPO | 说明 |
|---|---|---|---|
| `learning_rate` | 1e-5 | **4e-8** | DPO 低 250 倍,防止遗忘 |
| `epochs` | 3 | **1** | DPO 只训 1 轮 |
| `batch_size` | 32 | **4** | chosen + rejected 拼接 |
| `beta` | — | **0.15** | 约束温度 |
| `from_weight` | pretrained | **full_sft** | 必须基于 SFT 模型 |
| ref model | 无 | **冻结副本** | 显存 ×2 |
| `requires_grad` | 全部 True | policy True, ref **False** | — |

> 一句话:**DPO = 在 SFT 模型上轻轻推一把,让它在偏好对里选对边,同时别走太远。**